Coded: 14/9/2026

In [1]:


import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed(SEED)

Using device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


### Heston Simulation Function

In [2]:


def heston_model_sim(S0, v0, rho, kappa, theta, sigma, T, N, M):
    """
    Simulate Heston model paths.

    Returns:
        S: asset prices, shape (N+1, M)
        v: variance process, shape (N+1, M)
    """
    dt = T / N

    # Arrays for storing prices and variances
    S = np.full(shape=(N + 1, M), fill_value=S0)
    v = np.full(shape=(N + 1, M), fill_value=v0)

    # Manually construct correlated Brownian motions
    # Z1, Z2 are standard normal; Z2_corr = rho * Z1 + sqrt(1-rho^2) * Z3
    # This avoids np.random.multivariate_normal (which can be unstable/slow)
    Z1 = np.random.standard_normal((N, M))
    Z3 = np.random.standard_normal((N, M))
    Z2 = rho * Z1 + np.sqrt(1.0 - rho ** 2) * Z3

    for i in range(1, N + 1):
        S[i] = S[i - 1] * np.exp(
            (r - 0.5 * v[i - 1]) * dt
            + np.sqrt(v[i - 1] * dt) * Z1[i - 1]
        )
        v[i] = np.maximum(
            v[i - 1]
            + kappa * (theta - v[i - 1]) * dt
            + sigma * np.sqrt(v[i - 1] * dt) * Z2[i - 1],
            0.0
        )

    return S, v

In [3]:
#  Simulate Heston Paths (Diagnostic Run)

# --- Parameters ---
S0 = 100.0
T = 1.0
r = 0.02
N = 252

# Heston parameters
kappa = 3.0
theta = 0.20 ** 2
v0 = 0.25 ** 2
rho = -0.7
sigma = 0.6

# --- Start small for debugging ---
M = 100    # Number of paths. Increase to 10,000+ later.

start = time.time()
S_paths, v_paths = heston_model_sim(S0, v0, rho, kappa, theta, sigma, T, N, M)
print(f"Heston simulation took {time.time() - start:.3f} seconds")
print(f"S_paths shape: {S_paths.shape}")
print(f"v_paths shape: {v_paths.shape}")
print(f"S_paths range: [{S_paths.min():.2f}, {S_paths.max():.2f}]")
print(f"v_paths range: [{v_paths.min():.4f}, {v_paths.max():.4f}]")

# Sanity check: E[S_T] should be close to S0 * exp(r * T)
print(f"\nExpected E[S_T] under risk-neutral: {S0 * np.exp(r * T):.4f}")
print(f"Monte Carlo E[S_T]:                    {S_paths[-1].mean():.4f}")

Heston simulation took 0.010 seconds
S_paths shape: (253, 100)
v_paths shape: (253, 100)
S_paths range: [40.14, 139.63]
v_paths range: [0.0000, 0.3285]

Expected E[S_T] under risk-neutral: 102.0201
Monte Carlo E[S_T]:                    102.9307


### Define the JEPA Model

In [4]:
# ============================================================
# CELL 4: Build JEPA Training Dataset
# Context and target have the SAME sequence length.
# Target is the context shifted by 1 step (predict next window).
# ============================================================

context_len = 10
target_len = context_len  # <-- KEY CHANGE: same length as context
latent_dim = 16

features = np.stack([np.log(S_paths), v_paths], axis=-1)
print(f"Features shape: {features.shape}")

start = time.time()

# Number of windows per path
# Context starts at j, covers [j, j+context_len)
# Target starts at j+1, covers [j+1, j+1+context_len)
# Both must fit in [0, N+1)
num_windows_per_path = (N + 1) - context_len - 1

contexts = np.zeros((M * num_windows_per_path, context_len, 2), dtype=np.float32)
targets = np.zeros((M * num_windows_per_path, context_len, 2), dtype=np.float32)

idx = 0
for i in range(M):
    path_feats = features[:, i, :]  # (N+1, 2)
    for j in range(num_windows_per_path):
        contexts[idx] = path_feats[j : j + context_len]
        targets[idx]  = path_feats[j + 1 : j + 1 + context_len]
        idx += 1

print(f"Built {idx} windows in {time.time() - start:.3f} seconds")
print(f"Contexts shape: {contexts.shape}")
print(f"Targets shape:  {targets.shape}")

# --- Normalize features ---
feat_mean = contexts.reshape(-1, 2).mean(axis=0)
feat_std  = contexts.reshape(-1, 2).std(axis=0) + 1e-8
print(f"Feature mean: {feat_mean}")
print(f"Feature std:  {feat_std}")

contexts_norm = (contexts - feat_mean) / feat_std
targets_norm  = (targets  - feat_mean) / feat_std

contexts_tensor = torch.tensor(contexts_norm, dtype=torch.float32)
targets_tensor  = torch.tensor(targets_norm,  dtype=torch.float32)

dataset = TensorDataset(contexts_tensor, targets_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print(f"Number of batches: {len(dataloader)}")

Features shape: (253, 100, 2)
Built 24200 windows in 0.068 seconds
Contexts shape: (24200, 10, 2)
Targets shape:  (24200, 10, 2)
Feature mean: [4.609234   0.04478408]
Feature std:  [0.15474328 0.04775073]
Number of batches: 379


In [5]:
# ============================================================
# CELL 5: Define JEPA Model
# ============================================================

class Encoder(nn.Module):
    def __init__(self, input_dim, seq_len, latent_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim * seq_len, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = x.reshape(x.size(0), -1)
        return self.net(x)


class Predictor(nn.Module):
    def __init__(self, latent_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class JEPA(nn.Module):
    def __init__(self, input_dim, seq_len, latent_dim):
        super().__init__()
        self.context_encoder = Encoder(input_dim, seq_len, latent_dim)
        self.target_encoder = Encoder(input_dim, seq_len, latent_dim)
        self.predictor = Predictor(latent_dim)

        # Initialize target encoder weights = context encoder weights
        self.target_encoder.load_state_dict(self.context_encoder.state_dict())

        # Target encoder is not trained by gradient descent
        for p in self.target_encoder.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target_encoder(self, ema_decay):
        for p_q, p_k in zip(self.context_encoder.parameters(),
                            self.target_encoder.parameters()):
            p_k.data = p_k.data * ema_decay + p_q.data * (1.0 - ema_decay)

    def forward(self, context, target):
        z_ctx = self.context_encoder(context)
        with torch.no_grad():
            z_tgt = self.target_encoder(target)
        z_pred = self.predictor(z_ctx)
        loss = nn.functional.mse_loss(z_pred, z_tgt)
        return loss, z_ctx, z_tgt

### Pre-train the JEPA Model

In [6]:
# ============================================================
# CELL 6: Pre-train JEPA
# ============================================================

jepa_model = JEPA(input_dim=2, seq_len=context_len, latent_dim=latent_dim).to(device)

# Only optimize context encoder + predictor
params_to_optimize = list(jepa_model.context_encoder.parameters()) + \
                     list(jepa_model.predictor.parameters())
jepa_optimizer = optim.Adam(params_to_optimize, lr=1e-3)

ema_decay = 0.99
num_jepa_epochs = 20

print("Starting JEPA pre-training...")
start = time.time()

for epoch in range(num_jepa_epochs):
    jepa_model.train()
    total_loss = 0.0
    num_batches = 0

    for context, target in dataloader:
        context = context.to(device)
        target = target.to(device)

        jepa_optimizer.zero_grad()
        loss, _, _ = jepa_model(context, target)
        loss.backward()
        jepa_optimizer.step()
        jepa_model.update_target_encoder(ema_decay)

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:3d}/{num_jepa_epochs}, Loss: {avg_loss:.6f}")

print(f"JEPA training took {time.time() - start:.2f} seconds")
print("JEPA pre-training finished.")

Starting JEPA pre-training...
Epoch   1/20, Loss: 0.001771
Epoch   5/20, Loss: 0.023330
Epoch  10/20, Loss: 0.019770
Epoch  15/20, Loss: 0.022348
Epoch  20/20, Loss: 0.021203
JEPA training took 47.40 seconds
JEPA pre-training finished.


### Deep Hedging with the JEPA State

In [7]:
# ============================================================
# CELL 7: Hedging Policy Network
# ============================================================

class HedgingPolicy(nn.Module):
    def __init__(self, state_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state):
        return self.net(state)

In [8]:
# ============================================================
# CELL 8: Vectorized Deep Hedging Training Loop
# ============================================================

# --- Configuration ---
K = S0                    # At-the-money strike
cost_rate = 0.001         # Proportional transaction cost
num_hedging_epochs = 30
batch_size = 32           # Number of paths per gradient step

# --- Prepare data tensors ---
# features_tensor: (N+1, M, 2)  -- normalized features
features_norm = (features - feat_mean) / feat_std
features_tensor = torch.tensor(features_norm, dtype=torch.float32, device=device)

# S_tensor: (N+1, M) -- raw prices (NOT normalized)
S_tensor = torch.tensor(S_paths, dtype=torch.float32, device=device)

# Freeze JEPA
jepa_model.eval()

# --- Initialize policy ---
state_dim = latent_dim + 3   # z_ctx + [t/T, S/K, prev_delta]
policy = HedgingPolicy(state_dim).to(device)
policy_optimizer = optim.Adam(policy.parameters(), lr=1e-3)

print(f"Starting Deep Hedging training with {M} paths, "
      f"{num_hedging_epochs} epochs, batch_size={batch_size}")

start = time.time()

for epoch in range(num_hedging_epochs):
    # Shuffle path indices
    perm = torch.randperm(M, device=device)
    epoch_loss = 0.0
    num_batches = 0

    for batch_start in range(0, M, batch_size):
        idx = perm[batch_start : batch_start + batch_size]
        B = idx.shape[0]

        # Gather this batch's data
        S_batch = S_tensor[:, idx]           # (N+1, B)
        feat_batch = features_tensor[:, idx] # (N+1, B, 2)

        policy_optimizer.zero_grad()

        # ---- Recurrent hedge simulation ----
        prev_delta = torch.zeros(B, device=device)    # delta_{-1} = 0
        cumulative_cost = torch.zeros(B, device=device)
        gains = torch.zeros(B, device=device)

        # Iterate over time steps from t = context_len to N-1
        for t in range(context_len, N):
            # 1. Context window: last `context_len` steps up to time t-1
            #    feat_batch[t-context_len:t] has shape (context_len, B, 2)
            ctx = feat_batch[t - context_len : t].permute(1, 0, 2)  # (B, context_len, 2)

            with torch.no_grad():
                z_ctx = jepa_model.context_encoder(ctx)  # (B, latent_dim)

            # 2. Assemble state
            time_to_maturity = torch.full((B, 1), (N - t) / N, device=device)
            moneyness = (S_batch[t] / K).unsqueeze(1)                 # (B, 1)
            prev_delta_col = prev_delta.unsqueeze(1)                  # (B, 1)

            full_state = torch.cat(
                [z_ctx, time_to_maturity, moneyness, prev_delta_col], dim=1
            )  # (B, latent_dim + 3)

            # 3. New hedge
            new_delta = policy(full_state).squeeze(1)  # (B,)

            # 4. Accumulate gains: this position earns the return from t to t+1
            gains = gains + new_delta * (S_batch[t + 1] - S_batch[t])

            # 5. Transaction cost for changing position
            cumulative_cost = cumulative_cost + cost_rate * torch.abs(new_delta - prev_delta)

            prev_delta = new_delta

        # ---- Final liquidation cost at t = N ----
        cumulative_cost = cumulative_cost + cost_rate * torch.abs(prev_delta)

        # ---- Terminal P&L ----
        # P&L = gains - option_payoff - costs
        option_payoff = torch.relu(S_batch[N] - K)  # (B,)
        pnl = gains - option_payoff - cumulative_cost  # (B,)

        # ---- Loss: CVaR at 95% ----
        # Rockafellar-Uryasev: CVaR_alpha(X) = min_w [ w + (1/(1-alpha)) * E[max(-X - w, 0)] ]
        # We use alpha = 0.95 (worst 5% of losses)
        alpha = 0.95
        w = torch.tensor(0.0, device=device, requires_grad=True)
        # Loss is a one-sample estimate; we optimize w jointly with policy
        # In a full implementation, w is a learned parameter. For simplicity,
        # we use the batch quantile as w (this is a common approximation).
        with torch.no_grad():
            w_val = torch.quantile(-pnl, alpha).detach()
        losses = torch.clamp(-pnl - w_val, min=0.0)
        cvar_loss = w_val + losses.mean() / (1.0 - alpha)

        # ---- Backward ----
        cvar_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        policy_optimizer.step()

        epoch_loss += cvar_loss.item()
        num_batches += 1

    avg_loss = epoch_loss / max(num_batches, 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Hedging epoch {epoch + 1:3d}/{num_hedging_epochs}, "
              f"CVaR loss: {avg_loss:.4f}")

print(f"Deep Hedging training took {time.time() - start:.2f} seconds")
print("Training finished.")

Starting Deep Hedging training with 100 paths, 30 epochs, batch_size=32
Hedging epoch   1/30, CVaR loss: 23.8178
Hedging epoch   5/30, CVaR loss: 14.7614
Hedging epoch  10/30, CVaR loss: 14.9601
Hedging epoch  15/30, CVaR loss: 14.1492
Hedging epoch  20/30, CVaR loss: 13.2357
Hedging epoch  25/30, CVaR loss: 12.8808
Hedging epoch  30/30, CVaR loss: 13.4583
Deep Hedging training took 83.92 seconds
Training finished.


In [ ]:
# ============================================================
# CELL 9: Evaluate on Training Paths (Sanity Check)
# ============================================================

policy.eval()
jepa_model.eval()

with torch.no_grad():
    S_batch = S_tensor                     # (N+1, M)
    feat_batch = features_tensor
    B = M

    prev_delta = torch.zeros(B, device=device)
    cumulative_cost = torch.zeros(B, device=device)
    gains = torch.zeros(B, device=device)

    for t in range(context_len, N):
        ctx = feat_batch[t - context_len : t].permute(1, 0, 2)
        z_ctx = jepa_model.context_encoder(ctx)

        time_to_maturity = torch.full((B, 1), (N - t) / N, device=device)
        moneyness = (S_batch[t] / K).unsqueeze(1)
        prev_delta_col = prev_delta.unsqueeze(1)
        full_state = torch.cat([z_ctx, time_to_maturity, moneyness, prev_delta_col], dim=1)

        new_delta = policy(full_state).squeeze(1)
        gains = gains + new_delta * (S_batch[t + 1] - S_batch[t])
        cumulative_cost = cumulative_cost + cost_rate * torch.abs(new_delta - prev_delta)
        prev_delta = new_delta

    cumulative_cost = cumulative_cost + cost_rate * torch.abs(prev_delta)
    option_payoff = torch.relu(S_batch[N] - K)
    pnl = gains - option_payoff - cumulative_cost

    print(f"Terminal P&L statistics (JEPA-hedged):")
    print(f"  Mean:  {pnl.mean().item():.4f}")
    print(f"  Std:   {pnl.std().item():.4f}")
    print(f"  VaR95: {torch.quantile(-pnl, 0.95).item():.4f}")
    print(f"  CVaR95: {(torch.quantile(-pnl, 0.95) + torch.clamp(-pnl - torch.quantile(-pnl, 0.95), min=0).mean() / 0.05).item():.4f}")

# --- Black-Scholes delta hedge baseline ---
from scipy.stats import norm

def bs_delta(S, K, T, r, sigma):
    if T <= 0:
        return 1.0 if S > K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)

# --- FIX: use realized average vol from simulated paths ---
sigma_bs = float(np.sqrt(v_paths.mean()))
print(f"Using sigma_bs = {sigma_bs:.4f} for BS delta hedge")

with torch.no_grad():
    prev_delta = torch.zeros(M, device=device)
    cumulative_cost = torch.zeros(M, device=device)
    gains = torch.zeros(M, device=device)

    for t in range(context_len, N):
        S_np = S_batch[t].cpu().numpy()
        tau = (N - t) / N
        deltas_np = np.array([bs_delta(s, K, tau, r, sigma_bs) for s in S_np])
        new_delta = torch.tensor(deltas_np, dtype=torch.float32, device=device)

        gains = gains + new_delta * (S_batch[t + 1] - S_batch[t])
        cumulative_cost = cumulative_cost + cost_rate * torch.abs(new_delta - prev_delta)
        prev_delta = new_delta

    cumulative_cost = cumulative_cost + cost_rate * torch.abs(prev_delta)
    option_payoff = torch.relu(S_batch[N] - K)
    pnl_bs = gains - option_payoff - cumulative_cost

    print(f"\nTerminal P&L statistics (BS delta hedge):")
    print(f"  Mean:  {pnl_bs.mean().item():.4f}")
    print(f"  Std:   {pnl_bs.std().item():.4f}")
    print(f"  VaR95: {torch.quantile(-pnl_bs, 0.95).item():.4f}")
    print(f"  CVaR95: {(torch.quantile(-pnl_bs, 0.95) + torch.clamp(-pnl_bs - torch.quantile(-pnl_bs, 0.95), min=0).mean() / 0.05).item():.4f}")

Terminal P&L statistics (JEPA-hedged):
  Mean:  -7.8031
  Std:   2.9849
  VaR95: 12.1731
  CVaR95: 12.7730
Using sigma_bs = 0.2122 for BS delta hedge

Terminal P&L statistics (BS delta hedge):
  Mean:  -8.0737
  Std:   3.1572
  VaR95: 12.5767
  CVaR95: 15.1934


: 

In [ ]:
# ============================================================
# CELL 10: Compare P&L Distributions
# ============================================================

pnl_jepa = pnl.cpu().numpy()
pnl_bs_np = pnl_bs.cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pnl_jepa, bins=50, alpha=0.5, label="JEPA-hedged", density=True)
ax.hist(pnl_bs_np, bins=50, alpha=0.5, label="BS delta", density=True)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Terminal P&L")
ax.set_ylabel("Density")
ax.set_title("Hedging P&L Distribution: JEPA vs Black-Scholes Delta")
ax.legend()
plt.tight_layout()
plt.show()